# StructuredOutputParser → TypedDict + `with_structured_output()`

`StructuredOutputParser` / `ResponseSchema`는 LangChain v1에서 **`langchain-classic`** 패키지로 이동한 레거시 API입니다
(`from langchain.output_parsers import ...` 는 v1에서 동작하지 않습니다).

책에서 이 파서를 쓴 목적은 "**Pydantic 없이 key/value 형태의 dict**를 받는 것"이었습니다. 현재는 같은 목적을 **TypedDict 스키마 + `with_structured_output()`** 으로 달성합니다. 결과가 `dict`로 반환되고, 부분 결과 스트리밍도 됩니다.

**로컬 모델에 대하여**: 책에서는 로컬 모델에서 Pydantic 파서가 자주 실패하므로 이 파서를 대안으로 소개했습니다. 현재는 `langchain-ollama`의 `ChatOllama`도 `with_structured_output()`(기본 `method="json_schema"`, Ollama의 `format` 파라미터 사용)을 지원하므로, 로컬 모델에서도 같은 코드를 쓸 수 있습니다.

In [ ]:
# 최초 1회 설치 (LangChain v1 기준)
# %pip install -qU langchain langchain-openai langchain-classic python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # .env 의 OPENAI_API_KEY, LANGSMITH_API_KEY 를 불러옵니다.

# LangSmith 추적: 별도 헬퍼 없이 환경변수만 설정하면 자동으로 활성화됩니다.
if os.getenv("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ.setdefault("LANGSMITH_PROJECT", "CH03-OutputParser")

In [ ]:
from langchain.chat_models import init_chat_model

# 공급자 중립적인 모델 초기화 ("공급자:모델명")
# 다른 모델로 바꾸려면 문자열만 교체하면 됩니다. 예) "anthropic:claude-sonnet-4-5", "ollama:llama3.1"
llm = init_chat_model("openai:gpt-4.1-mini", temperature=0)

## 1. 스키마 정의 (ResponseSchema → TypedDict)

`ResponseSchema(name=..., description=...)` 목록 대신 `TypedDict`의 필드와 `Annotated` 설명을 사용합니다.

In [ ]:
from typing_extensions import Annotated, TypedDict


class AnswerWithSource(TypedDict):
    """사용자 질문에 대한 답변과 출처"""

    answer: Annotated[str, ..., "사용자의 질문에 대한 답변"]
    source: Annotated[
        str, ..., "사용자의 질문에 답하기 위해 사용된 `출처`, `웹사이트주소` 이여야 합니다."
    ]

## 2. 체인 구성

형식 지침(`format_instructions`)을 프롬프트에 넣을 필요가 없어 프롬프트가 단순해집니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "answer the users question as best as possible."),
        ("human", "{question}"),
    ]
)

chain = prompt | llm.with_structured_output(AnswerWithSource)

result = chain.invoke({"question": "대한민국의 수도는 어디인가요?"})
print(type(result))
result

## 3. 스트리밍

TypedDict 스키마는 필드가 채워지는 과정이 부분 dict로 스트리밍됩니다. (책의 파서는 완성된 결과만 출력했습니다.)

In [ ]:
for s in chain.stream({"question": "세종대왕의 업적은 무엇인가요?"}):
    print(s)

## 4. 로컬 모델에서 사용하기 (선택)

Ollama가 설치되어 있고 모델을 받아 둔 경우 아래 주석을 풀어 실행해 보세요. 체인 코드는 동일합니다.

In [ ]:
# %pip install -qU langchain-ollama
# local_llm = init_chat_model("ollama:llama3.1", temperature=0)
# local_chain = prompt | local_llm.with_structured_output(AnswerWithSource)
# local_chain.invoke({"question": "대한민국의 수도는 어디인가요?"})

## (참고) 레거시 API를 꼭 써야 하는 경우

기존 코드를 유지보수해야 한다면 `langchain-classic`에서 import합니다. 신규 코드에는 권장하지 않습니다.

In [ ]:
from langchain_classic.output_parsers import ResponseSchema, StructuredOutputParser

legacy_parser = StructuredOutputParser.from_response_schemas(
    [
        ResponseSchema(name="answer", description="사용자의 질문에 대한 답변"),
        ResponseSchema(name="source", description="답변에 사용된 출처 또는 웹사이트 주소"),
    ]
)

legacy_prompt = ChatPromptTemplate.from_template(
    "answer the users question as best as possible.\n{format_instructions}\n{question}"
).partial(format_instructions=legacy_parser.get_format_instructions())

(legacy_prompt | llm | legacy_parser).invoke({"question": "대한민국의 수도는 어디인가요?"})